# Shared geographic plot controls

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://www.dropbox.com/scl/fo/s22x9phl0hldiakn8nbuz/ABKfxHBaak5ra3eBGkNFWMM?rlkey=igpo7qi07oz5tfgjki317o79t&amp;st=gcxkicnc&amp;dl=1">Download tutorial data</a></div>

`GeoPlot` and `DotDensityPlot` share controls for outlines, focus, labels, colorbars,
and output. Each example below changes one behavior.


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

from gerrytools.plotting import GeoPlot, LabelOptions

precincts = gpd.read_file(Path("data/ga_2016_precincts.gpkg"))
precincts[["BVAP", "VAP"]] = precincts[["BVAP", "VAP"]].apply(pd.to_numeric)
precincts["BVAP_SHARE"] = precincts["BVAP"].div(precincts["VAP"])

## Outlines

`add_outline_layer()` accepts the base GeoDataFrame, a separate geographic source, or a
boolean mask. Supplying `dissolve_column=` converts finer source units into the boundaries of
a larger geography before drawing.


In [ ]:
outlines = GeoPlot(precincts)
outlines.add_outline_layer(dissolve_column="CTYNAME")
outlines.show()

## Focus

`focus_axes()` changes the axes limits without removing any layer data. Its default padding is
two percent of the selected geometry's width and height; `pad=` accepts one value, separate x
and y values, or four side-specific values.


In [ ]:
metro_mask = precincts["CTYNAME"].isin(["Clayton", "Cobb", "DeKalb", "Fulton", "Gwinnett"])
focused = GeoPlot(precincts)
focused.add_districting_plan_layer("CD")
focused.focus_axes(geometry_mask=metro_mask)
focused.show()

## Labels

District and outline layers can calculate one label position per dissolved geography. Named
styles cover the common treatments: `"halo"`, `"badge"`, `"ink"`, `"plain"`, and
`"tag"`. Start with a named label style, then use `LabelOptions` only for adjustments that
the style does not cover.


In [ ]:
labeled = GeoPlot(precincts)
labeled.add_districting_plan_layer(
    "CD",
    dissolve=True,
    show_labels=True,
    label_options=LabelOptions(label_style="badge"),
)
labeled.show()

## Colorbars

A choropleth returns a layer handle. `show_colorbar=True` attaches a colorbar immediately;
`add_colorbar(layer)` can attach one later to the same scale. This matters when a map contains
several continuous layers and only selected layers need keys.


In [ ]:
shares = GeoPlot(precincts)
shares.add_choropleth_layer("BVAP_SHARE", show_colorbar=True)
shares.show()

## Output

`show()` displays the current map and `save()` writes the same rendered state. Pass `title=`
when constructing the plot or assign `.title` later; `set_title_style()` provides the same
styling controls as the statistical builders. SVG and PDF preserve vector boundaries for
reports; PNG is useful when the final destination requires a raster image. The builder's `.ax`
remains available for Matplotlib-level composition.


In [ ]:
shares.save("bvap-share.svg")

## Advanced label placement

`LabelOptions` can adjust individual labels without replacing the named label style. The
`adjustments` mapping uses the displayed label as its key and an `(x, y)` offset as its value.
Offsets are measured in the map's coordinate units, so their useful magnitude depends on the
active projected coordinate reference system.


In [ ]:
adjusted = GeoPlot(precincts)
adjusted.add_districting_plan_layer(
    "CD",
    dissolve=True,
    show_labels=True,
    label_options=LabelOptions(
        label_style="badge",
        adjustments={5: (-60_000, -25_000), 6: (0, 25_000)},
    ),
)
adjusted.show()

## Related

- [Geographic plot formats](geo.ipynb)
- [Advanced geographic workflow](workflow.ipynb)
- [Plotting API](../../../api/plotting.rst)
